# 05 — MapBiomas: uso do solo e universo canavieiro final

Constrói o painel MapBiomas (uso e cobertura do solo) e — integrando com PAM (`04`) e ANP (`02`) — define o **universo canavieiro final** do paper pela união dos 3 critérios (§3.3 v2.2).

**Decisões metodológicas:**
- Critério MapBiomas: `share_cana > 5%` em algum ano 2015-2019.
- Universo canavieiro final = união de **PAM ∪ MB ∪ ANP**.
- Restrição ao Centro-Sul (6 UFs).
- 2 correções ortográficas conhecidas MapBiomas → IBGE (`PINGO-D'AGUA → PINGO D'AGUA`, `SANTO ANTONIO DE LEVERGER → SANTO ANTONIO DO LEVERGER`).

**Outputs em `data/interim/`:**
- `mapbiomas_panel.csv` — painel limpo (~30k linhas: 2.363 munis × 13 anos)
- `mapbiomas_canavieiro_baseline.csv` — munis com share > 5% baseline (~649)
- `universo_canavieiro_final.csv` — UNIÃO dos 3 critérios (~842)

**Tempo esperado:** ~30 segundos.

In [1]:
# Setup portável — resolve a raiz do repositório sem depender do Google Drive.
# Para executar a partir do Drive, defina antes: os.environ["RENOVABIO_BASE_DIR"] = "<caminho>"
# Para executar a partir do Drive, defina antes de rodar esta célula:
import os
import sys
from pathlib import Path

if os.environ.get("RENOVABIO_BASE_DIR"):
    BASE_DIR = Path(os.environ["RENOVABIO_BASE_DIR"]).expanduser().resolve()
else:
    BASE_DIR = Path.cwd().resolve()
    while not (BASE_DIR / "requirements.txt").exists() and BASE_DIR != BASE_DIR.parent:
        BASE_DIR = BASE_DIR.parent

if str(BASE_DIR) not in sys.path:
    sys.path.insert(0, str(BASE_DIR))
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')

Mounted at /content/drive


In [2]:
# Reload módulos
import importlib
from pipeline import config, normalize, io, mapbiomas
importlib.reload(config); importlib.reload(normalize)
importlib.reload(io); importlib.reload(mapbiomas)

from pipeline.config import PARAMS, interim, out_pre
from pipeline.mapbiomas import (
    run_mapbiomas_pipeline, MAPBIOMAS_IBGE_FIXES,
)
print('✓ módulos carregados')
print(f'  Correções ortográficas registradas: {len(MAPBIOMAS_IBGE_FIXES)}')
for k, v in MAPBIOMAS_IBGE_FIXES.items():
    print(f'    {k!r:50s} → {v!r}')

✓ módulos carregados
  Correções ortográficas registradas: 2
    "PINGO-D'AGUA|MG"                                  → "PINGO D'AGUA|MG"
    'SANTO ANTONIO DE LEVERGER|MT'                     → 'SANTO ANTONIO DO LEVERGER|MT'


In [3]:
# Carrega crosswalk + outputs do PAM e ANP
cw = pd.read_csv(interim('crosswalk_centrosul.csv'), dtype={'geocode': str})
print(f'Crosswalk: {cw.shape}')

canavieiros_pam = pd.read_csv(
    interim('pam_canavieiro_baseline.csv'), dtype={'geocode': str}
)
print(f'PAM canavieiros: {len(canavieiros_pam)} munis')

muni_treat_anp = pd.read_csv(
    interim('anp_muni_treat.csv'), dtype={'geocode': str}
)
print(f'ANP tratados: {len(muni_treat_anp)} munis')

Crosswalk: (2363, 5)
PAM canavieiros: 835 munis
ANP tratados: 194 munis


## Roda pipeline MapBiomas + união final

In [4]:
result = run_mapbiomas_pipeline(
    crosswalk=cw,
    canavieiros_pam=canavieiros_pam,
    muni_treat_anp=muni_treat_anp,
    save=True,
)

→ Lendo painel MapBiomas...
  raw: (31213, 51)

→ Limpando tipagem (áreas BR, shares com %)...
  cleaned: (31213, 51)

→ Resolvendo geocode via crosswalk...
  matched: 30719 cells | unmatched: 494 cells (38 munis)

→ Restringindo ao Centro-Sul...
  CS: (30719, 56) (2363 munis × 13 anos)

→ Construindo critério canavieiro MapBiomas (share > 5% baseline)...
  Canavieiros MapBiomas: 649 munis
  Por UF:
uf_cw
GO     47
MG     47
MS     18
MT      4
PR     97
SP    436

→ Construindo universo canavieiro final (união dos 3 critérios)...
  Total na união: 842 munis
  Por nº de critérios atendidos:
n_criterios_atendidos
1    176
2    496
3    170

  Por UF:
uf
GO     83
MG    124
MS     38
MT     22
PR    120
SP    455

→ Salvando...
  ✓ tudo salvo


## Inspeção do universo canavieiro final

In [5]:
universo = result['universo']
print(f'Universo canavieiro final: {len(universo)} munis')
print(f'\nValidação de integridade:')
print(f'  Sem município/UF nulo: '
      f'{(universo["municipio"].isna() | universo["uf"].isna()).sum() == 0}')
print(f'\nPor UF:')
print(universo.groupby("uf").size().sort_values(ascending=False).to_string())
print(f'\nPor nº de critérios atendidos:')
print(universo["n_criterios_atendidos"].value_counts().sort_index().to_string())

Universo canavieiro final: 842 munis

Validação de integridade:
  Sem município/UF nulo: True

Por UF:
uf
SP    455
MG    124
PR    120
GO     83
MS     38
MT     22

Por nº de critérios atendidos:
n_criterios_atendidos
1    176
2    496
3    170


In [6]:
# Cross-tab dos 3 critérios
ct = (universo
    .groupby(['is_canavieiro_pam', 'is_canavieiro_mb', 'is_canavieiro_anp'])
    .size().reset_index(name='n_munis'))
print('Cross-tab dos 3 critérios canavieiros:')
print(ct.to_string(index=False))
print()
print('Interpretação:')
print('  PAM=T, MB=T, ANP=T → núcleo duro (todas fontes concordam)')
print('  PAM=T, MB=T, ANP=F → cana cultivada vendida pra fora ou usina não certificada')
print('  PAM=T, MB=F, ANP=F → cana esparsa (não dominante no MB)')
print('  PAM=F, MB=T, ANP=F → MB-only (vale auditar — pode ser muni pequeno)')
print('  PAM=T, MB=F, ANP=T → divergência interessante')

Cross-tab dos 3 critérios canavieiros:
 is_canavieiro_pam  is_canavieiro_mb  is_canavieiro_anp  n_munis
             False              True              False        7
              True             False              False      169
              True             False               True       24
              True              True              False      472
              True              True               True      170

Interpretação:
  PAM=T, MB=T, ANP=T → núcleo duro (todas fontes concordam)
  PAM=T, MB=T, ANP=F → cana cultivada vendida pra fora ou usina não certificada
  PAM=T, MB=F, ANP=F → cana esparsa (não dominante no MB)
  PAM=F, MB=T, ANP=F → MB-only (vale auditar — pode ser muni pequeno)
  PAM=T, MB=F, ANP=T → divergência interessante


In [7]:
# Casos PAM=False, MB=True, ANP=False — auditoria visual
mb_only = universo[
    (~universo['is_canavieiro_pam']) &
    (universo['is_canavieiro_mb']) &
    (~universo['is_canavieiro_anp'])
]
print(f'Casos MB-only ({len(mb_only)} munis):')
print(f'(share_cana MB > 5% mas area_colhida PAM <= 500ha em todos os anos baseline)')
print()
if len(mb_only) > 0:
    cols_show = ['municipio', 'uf', 'mb_share_cana_max_baseline',
                 'mb_area_cana_max_baseline']
    print(mb_only[cols_show].to_string(index=False))

Casos MB-only (7 munis):
(share_cana MB > 5% mas area_colhida PAM <= 500ha em todos os anos baseline)

             municipio uf  mb_share_cana_max_baseline  mb_area_cana_max_baseline
              Itaguari GO                    0.059897                 854.459637
Santo Antônio de Goiás GO                    0.055459                 748.802789
        Capela do Alto SP                    0.075544                1283.512703
              Holambra SP                    0.100122                 656.549458
               Jumirim SP                    0.126457                 716.500098
              Pereiras SP                    0.054096                1207.211307
              Pracinha SP                    0.086043                 542.440899


## Top municípios canavieiros do Centro-Sul

In [8]:
# Núcleo duro (3 critérios) — Top 25 por área PAM
nucleo = universo[universo['n_criterios_atendidos'] == 3].copy()
nucleo_top = nucleo.nlargest(25, 'area_colhida_max_baseline')
print(f'\nNúcleo duro (3 critérios atendidos): {len(nucleo)} munis')
print(f'\nTop 25 por área colhida máxima baseline:')
cols = ['municipio', 'uf', 'mb_share_cana_max_baseline',
        'area_colhida_max_baseline', 'n_usinas']
nucleo_top[[c for c in cols if c in nucleo_top.columns]]


Núcleo duro (3 critérios atendidos): 170 munis

Top 25 por área colhida máxima baseline:


,municipio,uf,mb_share_cana_max_baseline,area_colhida_max_baseline,n_usinas
123,Morro Agudo,SP,0.755323,99000.0,2.0
46,Rio Brilhante,MS,0.253934,98002.0,3.0
43,Nova Alvorada do Sul,MS,0.230053,94925.0,1.0
37,Uberaba,MG,0.245659,84000.0,2.0
13,Quirinópolis,GO,0.230166,74396.0,1.0
25,Frutal,MG,0.360185,62006.0,2.0
99,Guaíra,SP,0.529668,62000.0,3.0
41,Costa Rica,MS,0.115348,61795.0,1.0
168,Valparaíso,SP,0.623737,58368.0,2.0
107,Jaboticabal,SP,0.785010,57550.0,1.0


## Resumo

Se o universo final saiu com **842 munis** distribuídos pelas 6 UFs (nada nulo), pipeline MapBiomas validado.

**Outputs salvos:**
- `data/interim/mapbiomas_panel.csv` — painel completo de uso do solo
- `data/interim/mapbiomas_canavieiro_baseline.csv` — apenas critério MB
- `data/interim/universo_canavieiro_final.csv` — **input principal de `09_assembly.ipynb`**

**Próximas camadas:**
- `06_sicar.ipynb` — outcomes H1c (CAR ativo, PRA, vegetação nativa)
- `07_psm_baseline.ipynb` — covariáveis baseline socioeconômicas
- `09_assembly.ipynb` — painel completo + canavieiro

**TODO de auditoria pré-submissão:**
- Os 7 casos `MB-only` (share > 5% mas área < 500ha) — auditar manualmente.
- Os 24 casos `PAM + ANP, sem MB` — investigar por que MapBiomas não detecta share > 5%.